# Argus — Connector Testing

Proves each data-source connection works in isolation, before any LangGraph agent logic touches them.

1. Prometheus (metrics)
2. Postgres — pg_stat_statements (DB performance) + pgvector (RAG embedding round-trip)
3. Grafana (dashboards / alert state)

In [13]:
import os
import requests
import psycopg2
from dotenv import load_dotenv

In [14]:
load_dotenv()

PROMETHEUS_URL = os.getenv("PROMETHEUS_URL", "http://localhost:9090")
GRAFANA_URL = os.getenv("GRAFANA_URL", "http://localhost:3000")
GRAFANA_API_KEY = os.getenv("GRAFANA_API_KEY")

In [15]:
PG_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "argus"),
    "user": os.getenv("POSTGRES_USER", "argus"),
    "password": os.getenv("POSTGRES_PASSWORD", "argus"),
}

In [16]:
def prometheus_query(promql: str):
    resp = requests.get(
        f"{PROMETHEUS_URL}/api/v1/query",
        params={"query": promql},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


result = prometheus_query("up")
for series in result["data"]["result"]:
    print(series["metric"]["job"], "->", series["value"][1])

node-exporter -> 1
otel-collector-otlp-metrics -> 1
otel-collector -> 1
cadvisor -> 1


In [17]:
result = prometheus_query("container_cpu_usage_seconds_total")
print(f"{len(result['data']['result'])} series returned")
if result["data"]["result"]:
    print(result["data"]["result"][0])

13 series returned
{'metric': {'__name__': 'container_cpu_usage_seconds_total', 'cpu': 'total', 'id': '/', 'instance': 'cadvisor:8080', 'job': 'cadvisor'}, 'value': [1786352729.007, '402.408']}


In [18]:
conn = psycopg2.connect(**PG_CONFIG)

with conn.cursor() as cur:
    cur.execute("SELECT extname, extversion FROM pg_extension;")
    print("Installed extensions:")
    for row in cur.fetchall():
        print(" -", row)

Installed extensions:
 - ('plpgsql', '1.0')
 - ('vector', '0.8.6')
 - ('pg_stat_statements', '1.10')


In [19]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT query, calls, total_exec_time, mean_exec_time
        FROM pg_stat_statements
        ORDER BY total_exec_time DESC
        LIMIT 5;
        """
    )
    for row in cur.fetchall():
        print(row)

('CREATE DATABASE "argus"', 1, 27.65832, 27.65832)
('CREATE TABLE IF NOT EXISTS incident_postmortems_test (\n            id SERIAL PRIMARY KEY,\n            summary TEXT,\n            embedding VECTOR(3)\n        )', 2, 17.905107, 8.9525535)
('CREATE EXTENSION IF NOT EXISTS vector', 1, 9.220665, 9.220665)
('CREATE EXTENSION IF NOT EXISTS pg_stat_statements', 1, 5.005194, 5.005194)
('DROP TABLE IF EXISTS incident_postmortems_test', 1, 3.097762, 3.097762)


In [20]:
with conn.cursor() as cur:
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS incident_postmortems_test (
            id SERIAL PRIMARY KEY,
            summary TEXT,
            embedding VECTOR(3)
        );
        """
    )
    cur.execute(
        "INSERT INTO incident_postmortems_test (summary, embedding) VALUES (%s, %s)",
        ("Test incident: DB connection pool exhausted", "[0.1, 0.2, 0.3]"),
    )
    conn.commit()

    cur.execute(
        "SELECT summary, embedding <-> %s AS distance FROM incident_postmortems_test ORDER BY distance LIMIT 3",
        ("[0.1, 0.2, 0.29]",),
    )
    for row in cur.fetchall():
        print(row)

with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS incident_postmortems_test;")
    conn.commit()

conn.close()

('Test incident: DB connection pool exhausted', 0.010000020246350199)


In [21]:
if not GRAFANA_API_KEY:
    print("GRAFANA_API_KEY not set yet — follow the steps above, then re-run this cell.")
else:
    headers = {"Authorization": f"Bearer {GRAFANA_API_KEY}"}
    resp = requests.get(f"{GRAFANA_URL}/api/health", headers=headers, timeout=10)
    print(resp.status_code, resp.json())

    resp = requests.get(f"{GRAFANA_URL}/api/datasources", headers=headers, timeout=10)
    print(resp.status_code, resp.json())

200 {'database': 'ok', 'version': '13.1.3', 'commit': '45a27d64b64a82d666b06aa5c5bb3521587edb0d'}
200 []
